In [1]:
import os

from langchain.embeddings import init_embeddings
from pymilvus import MilvusClient

# Milvus

In [2]:
# =========================
# 1. 基本配置
# =========================
MILVUS_URI = os.getenv("MILVUS_URI", "http://localhost:19530")  # Milvus 服务的连接地址
DB_NAME = "rag_tutorial"  # 自定义数据库名称
COLLECTION_NAME = "docs"  # 向量集合名称（类似于传统数据库的表）
KNOWLEDGE_FILE = "./knowledge.txt"  # 本地知识库文件路径

# BGE-M3 在 SiliconFlow / Milvus 文档中都是 1024 维
EMBED_MODEL_NAME = "qwen/qwen3-embedding-4b"  # 嵌入模型名称
EMBED_DIM = 2048  # BGE-M3 模型输出的向量维度固定为 1024

embedding_model_qwen = init_embeddings(
    model="openai:qwen/qwen3-embedding-4b",
    base_url=os.getenv("OPENROUTER_API_BASE"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    check_embedding_ctx_length=False,
    dimensions=2048,
)

In [10]:
# List database

client = MilvusClient(MILVUS_URI)
existing_dbs = client.list_databases()

print("databases:")
for db in existing_dbs:
    print(db, end=", ")

databases:
default, rag_tutorial, 

In [11]:
# Create database

if DB_NAME not in existing_dbs:
    client.create_database(DB_NAME)

client.use_database(DB_NAME)

In [ ]:
# Drop database

client.drop_database(DB_NAME)

In [21]:
# List collection

collections = client.list_collections()
print("collections:")
for collection in collections:
    print(collection, end=", ")

collections:
docs, 

In [18]:
# Create collection
client.create_collection(collection_name=COLLECTION_NAME,
                         dimension=EMBED_DIM,
                         metric_type="COSINE")

In [16]:
# Delete collection
client.drop_collection(collection_name=COLLECTION_NAME)

In [35]:
# Check metadata of collection

from rich import print as rprint

describe_collection = client.describe_collection(collection_name=COLLECTION_NAME)
rprint(describe_collection)

{
    'collection_name': 'docs',
    'auto_id': False,
    'num_shards': 1,
    'description': '',
    'fields': [
        {
            'field_id': 100,
            'name': 'id',
            'description': '',
            'type': <DataType.INT64: 5>,
            'params': {},
            'is_primary': True
        },
        {
            'field_id': 101,
            'name': 'vector',
            'description': '',
            'type': <DataType.FLOAT_VECTOR: 101>,
            'params': {'dim': 2048}
        }
    ],
    'functions': [],
    'aliases': [],
    'collection_id': 468148506956746210,
    'consistency_level': 2,
    'properties': {'timezone': 'UTC', 'namespace.sharding.enabled': 'false', 'max_field_id': '102'},
    'num_partitions': 1,
    'enable_dynamic_field': True,
    'enable_namespace': False,
    'created_timestamp': 468152827789443089,
    'update_timestamp': 468152827789443089
}

In [24]:
# 准备测试数据
texts = [
    "LangChain 是一个用于构建 LLM 应用的开发框架。",
    "Milvus 是一个适合 AI 应用的向量数据库。",
    "RAG 的核心是先检索相关知识，再让大模型生成答案。",
    "Docker Desktop 可以方便地在本地运行 Milvus Standalone。"
]

In [26]:
documents = embedding_model_qwen.embed_documents(texts=texts)
print(len(documents))
print(len(documents[0]))
print(documents[0][:5])

4
2048
[-0.0002698355237953365, -0.05321556329727173, -0.007803244050592184, 0.04221427068114281, -0.0009674103348515928]


In [33]:
# Prepare data

data = [
    {
        "id": i,
        "vector": documents[i],
        "text": texts[i],
        "source": "demo"
    } for i in range(len(documents))
]

In [34]:
# Insert data

upsert_result = client.upsert(collection_name=COLLECTION_NAME, data=data)
print(upsert_result)

{'upsert_count': 4, 'ids': [0, 1, 2, 3]}


In [36]:
# Flush collection manually

client.flush(collection_name=COLLECTION_NAME)

In [37]:
# Check collection stats

stats = client.get_collection_stats(collection_name=COLLECTION_NAME)
rprint(stats)

{'row_count': 4}

In [44]:
# Query data

iterator = client.query_iterator(collection_name=COLLECTION_NAME, filter="", output_fields=["*"])

i = 0
while True:
    rows = iterator.next()
    if rows is None or len(rows) == 0:
        break

    for row in rows:
        print(f"row {i}")
        print(f"id : {row["id"]},vector = {row["vector"][:5]},text = {row["text"]},source = {row["source"]}")
        i += 1

iterator.close()

row 0
id : 0,vector = [-0.0002698355237953365, -0.05321556329727173, -0.007803244050592184, 0.04221427068114281, -0.0009674103348515928],text = LangChain 是一个用于构建 LLM 应用的开发框架。,source = demo
row 1
id : 1,vector = [-0.000277012528385967, -0.025891704484820366, -0.016580894589424133, 0.04821213707327843, -0.0010203627170994878],text = Milvus 是一个适合 AI 应用的向量数据库。,source = demo
row 2
id : 2,vector = [-0.00030992753454484046, -0.03570365160703659, -0.010314388200640678, 0.02750503458082676, -0.0013554163742810488],text = RAG 的核心是先检索相关知识，再让大模型生成答案。,source = demo
row 3
id : 3,vector = [-0.00048344553215429187, -0.007801240775734186, -0.06240992620587349, 0.03940287604928017, -0.002107326639816165],text = Docker Desktop 可以方便地在本地运行 Milvus Standalone。,source = demo


In [42]:
# Get data

res = client.get(
    collection_name=COLLECTION_NAME,
    ids=[0, 1, 2]
)

print(len(res))

for i in range(len(res)):
    print(f"第{i + 1}条数据：")
    print(f"id : {res[i]["id"]},vector = {res[i]["vector"][:5]},text = {res[i]["text"]},source = {res[i]["source"]}")
    # print(res[i])


3
第1条数据：
id : 0,vector = [-0.0002698355237953365, -0.05321556329727173, -0.007803244050592184, 0.04221427068114281, -0.0009674103348515928],text = LangChain 是一个用于构建 LLM 应用的开发框架。,source = demo
第2条数据：
id : 1,vector = [-0.000277012528385967, -0.025891704484820366, -0.016580894589424133, 0.04821213707327843, -0.0010203627170994878],text = Milvus 是一个适合 AI 应用的向量数据库。,source = demo
第3条数据：
id : 2,vector = [-0.00030992753454484046, -0.03570365160703659, -0.010314388200640678, 0.02750503458082676, -0.0013554163742810488],text = RAG 的核心是先检索相关知识，再让大模型生成答案。,source = demo


In [45]:
# 相似度检索

query = "什么是向量数据库？"
query_vector = embedding_model_qwen.embed_query(query)

results = client.search(
    collection_name=COLLECTION_NAME,
    data=[query_vector],
    limit=3,
    output_fields=["text", "source", "id"]
)

for res in results[0]:
    print(res)

{'id': 1, 'distance': 0.6993657350540161, 'entity': {'id': 1, 'text': 'Milvus 是一个适合 AI 应用的向量数据库。', 'source': 'demo'}}
{'id': 3, 'distance': 0.45292073488235474, 'entity': {'id': 3, 'text': 'Docker Desktop 可以方便地在本地运行 Milvus Standalone。', 'source': 'demo'}}
{'id': 2, 'distance': 0.45179206132888794, 'entity': {'id': 2, 'text': 'RAG 的核心是先检索相关知识，再让大模型生成答案。', 'source': 'demo'}}
